In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "bohn2017information")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Bohn 2017_41598_2017_11400_MOESM3_ESM.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1)

df = df.rename(columns={"subject": "ape", 
    "species": "species_original"})
df['role']="subject"
df['study_id']="bohn2017information"


In [3]:
# df.trial.unique()

In [4]:
# df.columns

In [5]:

df['ape'] = df['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')
# df.columns
df.rename(columns={"ape": "participant", "group":"species_subgroup"}, inplace=True)

In [6]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365

In [7]:

bohn2017information_standardized = df[['study_id','year', 'month', 'day', 'participant', 'age_in_years',
        'sex','species',  'species_subgroup',  'session_cond','session_total', 'trial','condition_sep',
       'condition', 'toolends', 'exposure', 'bait', 'choice', 'perf', 'look',
       'food']]


comp_out_path_stand = os.path.join(out_pathway, 'bohn2017information_standardized.csv')
bohn2017information_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

In [8]:
names =bohn2017information_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
bohn2017information_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'bohn2017information_glossary.csv')
bohn2017information_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
